# Notebook 162: NLPのためのTransformer — アーキテクチャ完全解説

## Transformer for NLP: Complete Architecture Guide

---

### このノートブックの位置づけ

**言語モデルシリーズ** として、NLPの基盤技術であるTransformerアーキテクチャをゼロから実装しながら理解します。

| 項目 | 内容 |
|------|------|
| 難易度 | ★★★★☆ |
| 所要時間 | 120〜150分 |
| カテゴリ | 言語モデル |

### 学習目標

- [ ] Multi-Head Attentionをテキスト文脈で実装できる
- [ ] Sinusoidal/Learned/RoPEの位置エンコーディングを比較できる
- [ ] Encoder-only/Decoder-only/Encoder-Decoderの3分類を説明できる
- [ ] Pre-LN vs Post-LN の違いを理解できる
- [ ] 小規模Transformerでテキスト分類を実装できる

### 前提知識

- ✅ Notebook 95（ViT）
- ✅ Notebook 130（Temporal Attention）
- ✅ Notebook 152（文脈埋め込み）

---

## 目次

1. [環境セットアップ](#1-環境セットアップ)
2. [TransformerとNLP](#2-transformerとnlp)
3. [Scaled Dot-Product Attentionの実装](#3-scaled-dot-product-attentionの実装)
4. [Multi-Head Attentionの実装](#4-multi-head-attentionの実装)
5. [位置エンコーディング比較](#5-位置エンコーディング比較)
6. [3つのアーキテクチャ分類](#6-3つのアーキテクチャ分類)
7. [Layer Normalization](#7-layer-normalization)
8. [Feed-Forward Network](#8-feed-forward-network)
9. [小規模Transformerの実装](#9-小規模transformerの実装)
10. [まとめ・チートシート・よくある間違い](#10-まとめチートシートよくある間違い)
11. [自己評価クイズ](#11-自己評価クイズ)

---

## 1. 環境セットアップ

In [ ]:
# ============================================================
# Section 1: 環境セットアップ
# ============================================================
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import math
import warnings
warnings.filterwarnings('ignore')

# ------------------------------------------------------------
# 日本語フォント設定（環境に応じて調整）
# ------------------------------------------------------------
def setup_japanese_font():
    """日本語フォントをmatplotlibに設定する関数"""
    font_candidates = ['Hiragino Sans', 'Arial Unicode MS', 'IPAGothic', 'MS Gothic', 'sans-serif']
    plt.rcParams['font.family'] = font_candidates
    plt.rcParams['axes.unicode_minus'] = False
    plt.rcParams['figure.figsize'] = (10, 6)
    plt.rcParams['font.size'] = 11
    print("日本語フォント設定完了")

setup_japanese_font()

# ------------------------------------------------------------
# 再現性のためのシード
# ------------------------------------------------------------
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# Seaborn のスタイル設定
sns.set_style('whitegrid')

# デバイス設定
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"環境セットアップ完了")
print(f"  NumPy version: {np.__version__}")
print(f"  PyTorch version: {torch.__version__}")
print(f"  Device: {device}")

---

## 2. TransformerとNLP

### 2.1 Attention is All You Need の概要

2017年にVaswaniらが発表した論文 **"Attention is All You Need"** は、RNNやCNNを使わず、
**Self-Attention** のみに基づくアーキテクチャを提案し、NLPの世界を一変させました。

**Transformerの核心的なアイデア:**

1. **Self-Attention**: 入力系列の全トークン間の関係を並列に計算
2. **位置エンコーディング**: 系列順序の情報を明示的に注入
3. **Encoder-Decoder構造**: 入力の理解と出力の生成を分離

### 2.2 Self-Attentionの復習（テキスト文脈で）

Self-Attentionでは、各トークンが他の全トークンに「注目」し、文脈に応じた表現を得ます。

例: 「猫が**マット**の上に座った」
- 「マット」は「上に」「座った」に強く注目 → 場所の文脈を獲得
- 「猫」は「座った」に注目 → 主語-動詞の関係を獲得

### 2.3 Query / Key / Value の直感的理解

| 概念 | 直感 | 役割 |
|------|------|------|
| **Query (Q)** | 「何を知りたいか」 | 各トークンが発する質問 |
| **Key (K)** | 「自分は何を持っているか」 | 各トークンが提供する索引 |
| **Value (V)** | 「実際の中身」 | 各トークンが持つ情報 |

Attention = **Queryに最も関連するKeyを見つけ、対応するValueを重み付き合計する**

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

In [ ]:
# ============================================================
# Section 2: Query/Key/Value の直感的デモ
# ============================================================

# --- 簡単な例で Q, K, V の仕組みを理解する ---
# 3つの単語: ["猫", "が", "座った"]
# 各単語を4次元ベクトルで表現（手動設定）

print("=" * 60)
print("Query / Key / Value の直感的理解")
print("=" * 60)

# 埋め込みベクトル（手動設定）
words = ["猫", "が", "座った"]
embeddings = torch.tensor([
    [1.0, 0.0, 0.5, 0.2],  # 猫: 名詞的
    [0.0, 1.0, 0.0, 0.0],  # が: 助詞的
    [0.5, 0.0, 1.0, 0.8],  # 座った: 動詞的
])

print("\n入力埋め込み:")
for w, e in zip(words, embeddings):
    print(f"  {w}: {e.numpy()}")

# 重み行列（簡略化: 4x4）
W_Q = torch.tensor([[1, 0, 0, 0],
                     [0, 1, 0, 0],
                     [0, 0, 1, 0],
                     [0, 0, 0, 1]], dtype=torch.float32)
W_K = torch.tensor([[0, 1, 0, 0],
                     [1, 0, 0, 0],
                     [0, 0, 0, 1],
                     [0, 0, 1, 0]], dtype=torch.float32)
W_V = torch.eye(4)

# Q, K, V を計算
Q = embeddings @ W_Q  # 各単語の「質問」
K = embeddings @ W_K  # 各単語の「索引」
V = embeddings @ W_V  # 各単語の「情報」

print("\nQuery (Q = X @ W_Q):")
for w, q in zip(words, Q):
    print(f"  {w}: {q.numpy()}")

print("\nKey (K = X @ W_K):")
for w, k in zip(words, K):
    print(f"  {w}: {k.numpy()}")

# Attention Score = Q @ K^T
d_k = Q.shape[-1]
scores = Q @ K.T / math.sqrt(d_k)
print(f"\nAttention Scores (QK^T / sqrt({d_k})):")
print(scores.numpy().round(3))

# Softmax で正規化
attn_weights = F.softmax(scores, dim=-1)
print(f"\nAttention Weights (softmax):")
for i, w in enumerate(words):
    print(f"  {w} → {dict(zip(words, attn_weights[i].numpy().round(3)))}")

print("\n【解釈】")
print("  各行は、その単語が他の単語にどれだけ注目しているかを示す")
print("  例: '猫' が '座った' に高い重みを持つなら、主語-動詞の関係を捉えている")

---

## 3. Scaled Dot-Product Attentionの実装

In [ ]:
# ============================================================
# Section 3: Scaled Dot-Product Attention のスクラッチ実装
# ============================================================

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Scaled Dot-Product Attention の実装。
    
    Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) V
    
    Parameters:
        Q: Query テンソル [batch, seq_len_q, d_k]
        K: Key テンソル [batch, seq_len_k, d_k]
        V: Value テンソル [batch, seq_len_k, d_v]
        mask: マスクテンソル（Noneまたは [batch, seq_len_q, seq_len_k]）
              True/1 の位置がマスクされる（-inf に置換）
    
    Returns:
        output: Attention出力 [batch, seq_len_q, d_v]
        attn_weights: Attention重み [batch, seq_len_q, seq_len_k]
    """
    # d_k: Keyの次元数
    d_k = K.shape[-1]
    
    # Attention Score の計算: QK^T / sqrt(d_k)
    # Q: [B, L_q, d_k] @ K^T: [B, d_k, L_k] -> [B, L_q, L_k]
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    
    # マスクの適用（オプション）
    # マスクされた位置を -inf にすることで、softmax 後に 0 になる
    if mask is not None:
        scores = scores.masked_fill(mask == 1, float('-inf'))
    
    # Softmax で正規化 → Attention重み
    attn_weights = F.softmax(scores, dim=-1)
    
    # Value との重み付き和 → 出力
    output = torch.matmul(attn_weights, V)
    
    return output, attn_weights


# --- テスト ---
print("=" * 60)
print("Scaled Dot-Product Attention テスト")
print("=" * 60)

# バッチサイズ1、系列長5、次元8
batch_size = 1
seq_len = 5
d_k = 8

# ランダムな入力
torch.manual_seed(42)
Q_test = torch.randn(batch_size, seq_len, d_k)
K_test = torch.randn(batch_size, seq_len, d_k)
V_test = torch.randn(batch_size, seq_len, d_k)

output, weights = scaled_dot_product_attention(Q_test, K_test, V_test)

print(f"\nQ shape: {Q_test.shape}")
print(f"K shape: {K_test.shape}")
print(f"V shape: {V_test.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {weights.shape}")
print(f"\nAttention weights (各行の合計=1):")
print(f"  行の合計: {weights[0].sum(dim=-1).numpy().round(4)}")
print("\n各行がsoftmaxで正規化されているため、合計が1.0になる")

In [ ]:
# ============================================================
# Attention重みの可視化（テキストペア）
# ============================================================

# より意味のある例: 簡単な文でAttentionパターンを可視化
# 手動で設定した埋め込みを使用
tokens = ["私", "は", "東京", "で", "寿司", "を", "食べた"]
n_tokens = len(tokens)
d_model = 16

# ランダムだが再現可能な埋め込み
torch.manual_seed(42)
X = torch.randn(1, n_tokens, d_model)

# 線形変換（学習される重み行列のシミュレーション）
W_q = nn.Linear(d_model, d_model, bias=False)
W_k = nn.Linear(d_model, d_model, bias=False)
W_v = nn.Linear(d_model, d_model, bias=False)

with torch.no_grad():
    Q = W_q(X)
    K = W_k(X)
    V = W_v(X)

output, attn_weights = scaled_dot_product_attention(Q, K, V)

# Attention重みのヒートマップ
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    attn_weights[0].detach().numpy(),
    annot=True, fmt='.2f', cmap='YlOrRd',
    xticklabels=tokens, yticklabels=tokens,
    ax=ax, square=True, linewidths=0.5, linecolor='white'
)
ax.set_xlabel('Key（注目先）', fontsize=12)
ax.set_ylabel('Query（注目元）', fontsize=12)
ax.set_title('Scaled Dot-Product Attention 重み\n「私は東京で寿司を食べた」', fontsize=14)
plt.tight_layout()
plt.show()

print("【注意】この例はランダム初期化なので、意味的なパターンは偶然")
print("  学習後は、'食べた'→'寿司'、'東京'→'で' など意味的な注目が現れる")

In [ ]:
# ============================================================
# スケーリングの重要性の実験
# ============================================================

# d_k が大きくなると、ドット積の値が大きくなり、
# softmax が極端な分布（ほぼone-hot）になってしまう問題

print("=" * 60)
print("スケーリング (1/sqrt(d_k)) の重要性")
print("=" * 60)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

dims = [8, 64, 512]
torch.manual_seed(42)

for idx, d in enumerate(dims):
    # ランダムなQ, K
    Q_exp = torch.randn(1, 10, d)
    K_exp = torch.randn(1, 10, d)
    
    # スケーリングなし
    scores_raw = torch.matmul(Q_exp, K_exp.transpose(-2, -1))
    weights_no_scale = F.softmax(scores_raw, dim=-1)
    
    # スケーリングあり
    scores_scaled = scores_raw / math.sqrt(d)
    weights_scaled = F.softmax(scores_scaled, dim=-1)
    
    ax = axes[idx]
    
    # 各行のエントロピーを計算（均一な分布ほど高い）
    entropy_no_scale = -(weights_no_scale * torch.log(weights_no_scale + 1e-9)).sum(-1).mean().item()
    entropy_scaled = -(weights_scaled * torch.log(weights_scaled + 1e-9)).sum(-1).mean().item()
    max_entropy = math.log(10)  # 均一分布のエントロピー
    
    # Attention重みの分布をヒストグラムで表示
    ax.hist(weights_no_scale[0].detach().numpy().flatten(), bins=30, alpha=0.6,
            label=f'スケーリングなし\n(H={entropy_no_scale:.2f})', color='#e74c3c')
    ax.hist(weights_scaled[0].detach().numpy().flatten(), bins=30, alpha=0.6,
            label=f'スケーリングあり\n(H={entropy_scaled:.2f})', color='#3498db')
    ax.axvline(x=1/10, color='green', linestyle='--', linewidth=1.5, label='均一分布値')
    ax.set_xlabel('Attention重みの値', fontsize=11)
    ax.set_ylabel('頻度', fontsize=11)
    ax.set_title(f'd_k = {d}\n(最大エントロピー: {max_entropy:.2f})', fontsize=12)
    ax.legend(fontsize=9)

plt.suptitle('スケーリングの効果: d_k が大きいほどスケーリングが重要', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("\n【観察ポイント】")
print("  d_k=8:   スケーリングの有無でほとんど差がない")
print("  d_k=64:  スケーリングなしだとやや極端な分布に")
print("  d_k=512: スケーリングなしだとほぼone-hot（勾配消失の危険）")
print("\n  sqrt(d_k) で割ることで、次元数に関わらず適度な分布を維持できる")

---

## 4. Multi-Head Attentionの実装

### 4.1 なぜ複数のヘッドが必要か

単一のAttentionヘッドでは、1つの「観点」からしか注目パターンを学習できません。
Multi-Head Attentionでは、**複数の異なる射影空間**でAttentionを並列に計算し、
異なる言語的関係を同時に捉えます。

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, ..., \text{head}_h) W^O$$

$$\text{head}_i = \text{Attention}(Q W_i^Q, K W_i^K, V W_i^V)$$

In [ ]:
# ============================================================
# Section 4: Multi-Head Attention のスクラッチ実装
# ============================================================

class MultiHeadAttention(nn.Module):
    """
    Multi-Head Attention の実装。
    
    複数のAttentionヘッドを並列に計算し、
    異なる「観点」からの注目パターンを統合する。
    
    Parameters:
        d_model: モデルの次元数
        n_heads: Attentionヘッドの数
    """
    
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0, "d_model は n_heads で割り切れる必要がある"
        
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads  # 各ヘッドの次元数
        
        # 各ヘッド用の線形変換（まとめて1つの大きな行列で実装）
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        
        # 出力射影
        self.W_O = nn.Linear(d_model, d_model, bias=False)
        
        # Attention重みを保存（可視化用）
        self.attn_weights = None
    
    def forward(self, Q, K, V, mask=None):
        """
        Parameters:
            Q: [batch, seq_len_q, d_model]
            K: [batch, seq_len_k, d_model]
            V: [batch, seq_len_k, d_model]
            mask: [batch, 1, seq_len_q, seq_len_k] or None
        
        Returns:
            output: [batch, seq_len_q, d_model]
        """
        batch_size = Q.shape[0]
        
        # Step 1: 線形変換
        # [batch, seq_len, d_model] -> [batch, seq_len, d_model]
        Q = self.W_Q(Q)
        K = self.W_K(K)
        V = self.W_V(V)
        
        # Step 2: ヘッドに分割
        # [batch, seq_len, d_model] -> [batch, n_heads, seq_len, d_k]
        Q = Q.view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        
        # Step 3: 各ヘッドで Scaled Dot-Product Attention
        # scores: [batch, n_heads, seq_len_q, seq_len_k]
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 1, float('-inf'))
        
        attn_weights = F.softmax(scores, dim=-1)
        self.attn_weights = attn_weights.detach()  # 可視化用に保存
        
        # [batch, n_heads, seq_len_q, d_k]
        context = torch.matmul(attn_weights, V)
        
        # Step 4: ヘッドを結合
        # [batch, n_heads, seq_len, d_k] -> [batch, seq_len, d_model]
        context = context.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        
        # Step 5: 出力射影
        output = self.W_O(context)
        
        return output


# --- テスト ---
print("=" * 60)
print("Multi-Head Attention テスト")
print("=" * 60)

d_model = 64
n_heads = 4
seq_len = 7
batch_size = 1

mha = MultiHeadAttention(d_model=d_model, n_heads=n_heads)

# 入力（Self-Attentionの場合、Q=K=V=X）
torch.manual_seed(42)
X = torch.randn(batch_size, seq_len, d_model)

with torch.no_grad():
    output = mha(X, X, X)

print(f"\n入力 X shape: {X.shape}")
print(f"出力 shape: {output.shape}")
print(f"Attention weights shape: {mha.attn_weights.shape}")
print(f"  → [batch={batch_size}, heads={n_heads}, seq={seq_len}, seq={seq_len}]")
print(f"\nd_model={d_model}, n_heads={n_heads} → 各ヘッドの次元 d_k={d_model//n_heads}")

In [ ]:
# ============================================================
# ヘッドごとのAttentionパターン可視化
# ============================================================

# 各ヘッドが異なるパターンで注目することを確認
tokens = ["私", "は", "東京", "で", "寿司", "を", "食べた"]

fig, axes = plt.subplots(1, n_heads, figsize=(5 * n_heads, 5))

for head_idx in range(n_heads):
    ax = axes[head_idx]
    weights = mha.attn_weights[0, head_idx].numpy()
    
    sns.heatmap(
        weights, annot=True, fmt='.2f', cmap='Blues',
        xticklabels=tokens, yticklabels=tokens,
        ax=ax, square=True, linewidths=0.5, linecolor='white',
        vmin=0, vmax=0.5
    )
    ax.set_title(f'Head {head_idx + 1}', fontsize=13, fontweight='bold')
    if head_idx == 0:
        ax.set_ylabel('Query（注目元）', fontsize=11)
    ax.set_xlabel('Key（注目先）', fontsize=11)

plt.suptitle('Multi-Head Attention: 各ヘッドの注目パターン\n（ランダム初期化時）', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("【各ヘッドの役割（学習後の典型例）】")
print("  Head 1: 隣接トークンへの注目（局所的な文法関係）")
print("  Head 2: 主語-動詞の対応関係")
print("  Head 3: 修飾語-被修飾語の関係")
print("  Head 4: 文全体の要約的な注目")
print("\n※ 上図はランダム初期化時のため、これらのパターンは学習後に現れる")

In [ ]:
# ============================================================
# 各ヘッドが異なる言語的関係を捉える例（人工的デモ）
# ============================================================

# 学習済みモデルを模倣: 手動で設計したAttention重みで
# 各ヘッドが異なる言語的関係を捉えることをデモ

# 文: "The cat sat on the mat"
tokens_en = ["The", "cat", "sat", "on", "the", "mat"]
n = len(tokens_en)

# ヘッド1: 位置的近接（隣接トークンに注目）
head1 = torch.zeros(n, n)
for i in range(n):
    for j in range(n):
        head1[i, j] = math.exp(-abs(i - j))
head1 = head1 / head1.sum(dim=-1, keepdim=True)

# ヘッド2: 構文的関係（主語-動詞、前置詞-名詞）
head2 = torch.zeros(n, n)
head2[2, 1] = 1.0  # sat -> cat (動詞→主語)
head2[1, 2] = 1.0  # cat -> sat (主語→動詞)
head2[5, 3] = 1.0  # mat -> on (名詞→前置詞)
head2[3, 5] = 1.0  # on -> mat (前置詞→名詞)
for i in range(n):
    if head2[i].sum() == 0:
        head2[i, i] = 1.0
head2 = head2 / head2.sum(dim=-1, keepdim=True)

# ヘッド3: 冠詞-名詞の関係
head3 = torch.zeros(n, n)
head3[0, 1] = 1.0  # The -> cat
head3[1, 0] = 1.0  # cat -> The
head3[4, 5] = 1.0  # the -> mat
head3[5, 4] = 1.0  # mat -> the
for i in range(n):
    if head3[i].sum() == 0:
        head3[i, i] = 1.0
head3 = head3 / head3.sum(dim=-1, keepdim=True)

heads = [head1, head2, head3]
head_names = ["位置的近接", "構文的関係\n(主語-動詞)", "冠詞-名詞\nの対応"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (head, name) in enumerate(zip(heads, head_names)):
    ax = axes[idx]
    sns.heatmap(
        head.numpy(), annot=True, fmt='.2f', cmap='Purples',
        xticklabels=tokens_en, yticklabels=tokens_en,
        ax=ax, square=True, linewidths=0.5, linecolor='white',
        vmin=0, vmax=1.0
    )
    ax.set_title(f'Head {idx+1}: {name}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Key', fontsize=11)
    if idx == 0:
        ax.set_ylabel('Query', fontsize=11)

plt.suptitle('各ヘッドが捉える異なる言語的関係（人工的デモ）\n"The cat sat on the mat"', fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

print("【Multi-Head Attentionの利点】")
print("  複数のヘッドが異なる種類の関係を並列に学習できる")
print("  ・位置的な近接関係（n-gram的）")
print("  ・構文的な依存関係（主語-動詞など）")
print("  ・意味的な対応関係（共参照など）")

---

## 5. 位置エンコーディング比較

TransformerはSelf-Attentionのみで構成されるため、入力トークンの**順序情報**を持ちません。
位置エンコーディング（Positional Encoding）を加えることで、系列の順序を表現します。

3つの代表的な手法を実装・比較します:

1. **Sinusoidal PE** (Vaswani et al., 2017) — 固定的、三角関数ベース
2. **Learned PE** (BERT, GPT) — 学習可能なパラメータ
3. **RoPE** (Su et al., 2021) — 回転による相対位置エンコーディング

In [ ]:
# ============================================================
# Section 5: 位置エンコーディングの実装と比較
# ============================================================

# --- 5.1 Sinusoidal Positional Encoding ---

class SinusoidalPositionalEncoding(nn.Module):
    """
    正弦波ベースの位置エンコーディング（Attention is All You Need）。
    
    PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    
    特徴:
    - 固定（学習不要）
    - 任意の長さの系列に対応可能
    - 相対位置を線形変換で表現できる
    """
    
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        
        # 位置エンコーディング行列を事前計算
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        
        # 周波数の計算: 1 / 10000^(2i/d_model)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        
        # 偶数次元: sin、奇数次元: cos
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        # [1, max_len, d_model] に変形してバッファに登録
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        """x: [batch, seq_len, d_model]"""
        return x + self.pe[:, :x.size(1), :]


# --- 5.2 Learned Positional Encoding ---

class LearnedPositionalEncoding(nn.Module):
    """
    学習可能な位置エンコーディング（BERT, GPT で使用）。
    
    特徴:
    - データから最適な位置表現を学習
    - 最大系列長が固定される
    - 柔軟だが、訓練データに依存
    """
    
    def __init__(self, d_model, max_len=512):
        super().__init__()
        self.embedding = nn.Embedding(max_len, d_model)
    
    def forward(self, x):
        """x: [batch, seq_len, d_model]"""
        seq_len = x.size(1)
        positions = torch.arange(0, seq_len, device=x.device)
        return x + self.embedding(positions).unsqueeze(0)


# --- 5.3 RoPE (Rotary Position Embedding) ---

class RotaryPositionalEncoding(nn.Module):
    """
    RoPE (Rotary Position Embedding) の実装。
    
    数理:
    - ベクトルの隣接ペア (x_{2i}, x_{2i+1}) を、
      位置 m に応じた角度 m*theta_i で回転させる
    - theta_i = 10000^(-2i/d)
    
    特徴:
    - 相対位置情報をドット積に自然にエンコード
    - 任意長への外挿が可能
    - LLaMA, GPT-NeoX 等の現代モデルで採用
    """
    
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        self.d_model = d_model
        
        # 周波数の事前計算
        # theta_i = 10000^(-2i/d) for i = 0, 1, ..., d/2-1
        inv_freq = 1.0 / (10000 ** (torch.arange(0, d_model, 2).float() / d_model))
        self.register_buffer('inv_freq', inv_freq)
        
        # cos, sin のキャッシュ
        self._build_cache(max_len)
    
    def _build_cache(self, max_len):
        """cos/sinテーブルを事前計算"""
        t = torch.arange(max_len, dtype=torch.float)
        freqs = torch.outer(t, self.inv_freq)  # [max_len, d/2]
        # [max_len, d] に拡張（各周波数を2回繰り返す）
        emb = torch.cat([freqs, freqs], dim=-1)  # [max_len, d]
        self.register_buffer('cos_cached', emb.cos().unsqueeze(0))  # [1, max_len, d]
        self.register_buffer('sin_cached', emb.sin().unsqueeze(0))
    
    def _rotate_half(self, x):
        """ベクトルの前半と後半を入れ替えて符号反転"""
        d = x.shape[-1]
        x1 = x[..., :d // 2]
        x2 = x[..., d // 2:]
        return torch.cat([-x2, x1], dim=-1)
    
    def forward(self, x):
        """x: [batch, seq_len, d_model]"""
        seq_len = x.size(1)
        cos = self.cos_cached[:, :seq_len, :]
        sin = self.sin_cached[:, :seq_len, :]
        # RoPE: x * cos + rotate_half(x) * sin
        return x * cos + self._rotate_half(x) * sin


print("3つの位置エンコーディングの実装完了")
print("  1. SinusoidalPositionalEncoding (固定)")
print("  2. LearnedPositionalEncoding (学習可能)")
print("  3. RotaryPositionalEncoding (回転ベース)")

In [ ]:
# ============================================================
# Sinusoidal PE の波形パターン可視化
# ============================================================

d_model_pe = 64
max_len_pe = 100

# Sinusoidal PE を生成
sin_pe = SinusoidalPositionalEncoding(d_model_pe, max_len_pe)
pe_values = sin_pe.pe[0, :max_len_pe, :].numpy()  # [max_len, d_model]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# --- 左上: PE全体のヒートマップ ---
ax = axes[0, 0]
im = ax.imshow(pe_values.T, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xlabel('位置 (position)', fontsize=11)
ax.set_ylabel('次元 (dimension)', fontsize=11)
ax.set_title('Sinusoidal PE: 全体パターン', fontsize=13)
plt.colorbar(im, ax=ax, shrink=0.8)

# --- 右上: 異なる次元の波形 ---
ax = axes[0, 1]
dims_to_show = [0, 1, 4, 5, 20, 21, 60, 61]
colors_pe = plt.cm.viridis(np.linspace(0, 1, len(dims_to_show)))
for dim_idx, color in zip(dims_to_show, colors_pe):
    label = f'dim {dim_idx} ({"sin" if dim_idx % 2 == 0 else "cos"})'
    ax.plot(pe_values[:50, dim_idx], label=label, color=color, linewidth=1.5)
ax.set_xlabel('位置', fontsize=11)
ax.set_ylabel('PE値', fontsize=11)
ax.set_title('各次元の波形（周波数が異なる）', fontsize=13)
ax.legend(fontsize=8, loc='upper right', ncol=2)
ax.grid(True, alpha=0.3)

# --- 左下: 位置間のコサイン類似度 ---
ax = axes[1, 0]
pe_normed = pe_values / np.linalg.norm(pe_values, axis=1, keepdims=True)
pe_sim = pe_normed @ pe_normed.T
im2 = ax.imshow(pe_sim[:50, :50], aspect='auto', cmap='YlOrRd', vmin=-0.5, vmax=1.0)
ax.set_xlabel('位置', fontsize=11)
ax.set_ylabel('位置', fontsize=11)
ax.set_title('位置間のコサイン類似度\n（近い位置ほど類似度が高い）', fontsize=13)
plt.colorbar(im2, ax=ax, shrink=0.8)

# --- 右下: 距離と類似度の関係 ---
ax = axes[1, 1]
# 位置0からの距離と類似度
ref_pos = 0
similarities = pe_sim[ref_pos, :50]
ax.plot(range(50), similarities, 'o-', markersize=3, color='#e74c3c', linewidth=1.5)
ax.set_xlabel(f'位置0からの距離', fontsize=11)
ax.set_ylabel('コサイン類似度', fontsize=11)
ax.set_title('位置0との類似度\n（距離が増すと減衰しつつ振動）', fontsize=13)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("【Sinusoidal PE の特徴】")
print("  低次元: 長い周期の波 → 大まかな位置情報")
print("  高次元: 短い周期の波 → 細かい位置情報")
print("  近い位置ほどコサイン類似度が高い → 相対位置の情報を含む")

In [ ]:
# ============================================================
# RoPE の数理的直感の可視化
# ============================================================

# RoPE: 2次元の回転で理解する
# ベクトル (x1, x2) を角度 theta で回転:
# (x1*cos(theta) - x2*sin(theta), x1*sin(theta) + x2*cos(theta))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- 左: 2D回転の可視化 ---
ax = axes[0]
original = np.array([1.0, 0.5])
angles = np.arange(0, 8) * np.pi / 8  # 8つの位置

colors_rope = plt.cm.plasma(np.linspace(0, 0.9, len(angles)))
for i, theta in enumerate(angles):
    cos_t, sin_t = np.cos(theta), np.sin(theta)
    rotated = np.array([
        original[0] * cos_t - original[1] * sin_t,
        original[0] * sin_t + original[1] * cos_t
    ])
    ax.annotate('', xy=rotated, xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=colors_rope[i], lw=2))
    ax.text(rotated[0] * 1.15, rotated[1] * 1.15, f'pos={i}',
            fontsize=9, color=colors_rope[i], fontweight='bold', ha='center')

# 原点と円
circle = plt.Circle((0, 0), np.linalg.norm(original), fill=False,
                     linestyle='--', color='gray', alpha=0.5)
ax.add_patch(circle)
ax.set_xlim(-1.8, 1.8)
ax.set_ylim(-1.8, 1.8)
ax.set_aspect('equal')
ax.set_title('RoPE: 位置による回転\n（2次元ペアの回転）', fontsize=13)
ax.set_xlabel('次元 2i', fontsize=11)
ax.set_ylabel('次元 2i+1', fontsize=11)
ax.grid(True, alpha=0.3)
ax.axhline(y=0, color='k', linewidth=0.5)
ax.axvline(x=0, color='k', linewidth=0.5)

# --- 右: RoPEでの相対位置とドット積 ---
ax = axes[1]

# RoPEの重要な性質: <RoPE(q, m), RoPE(k, n)> は m-n にのみ依存
d_rope = 32
rope = RotaryPositionalEncoding(d_rope, max_len=100)

torch.manual_seed(42)
q_vec = torch.randn(1, 1, d_rope).expand(1, 50, d_rope)  # 同じqを全位置に
k_vec = torch.randn(1, 1, d_rope).expand(1, 50, d_rope)  # 同じkを全位置に

q_rope = rope(q_vec)  # 各位置でRoPE適用
k_rope = rope(k_vec)

# 位置0のqと各位置のkのドット積
dots = (q_rope[0, 0:1, :] @ k_rope[0].T).squeeze().detach().numpy()

ax.plot(range(50), dots, 'o-', markersize=3, color='#2ecc71', linewidth=1.5)
ax.set_xlabel('Key の位置 (n)', fontsize=11)
ax.set_ylabel('ドット積', fontsize=11)
ax.set_title('RoPE: Query(pos=0) と Key(pos=n) のドット積\n（相対位置 n に依存）', fontsize=13)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("【RoPE の核心】")
print("  ドット積 <RoPE(q, m), RoPE(k, n)> が相対位置 (m-n) にのみ依存する")
print("  → 絶対位置ではなく相対位置をエンコードする")
print("  → 訓練時より長い系列にも外挿可能（length extrapolation）")

In [ ]:
# ============================================================
# 3つの位置エンコーディング手法の比較表
# ============================================================

print("=" * 80)
print("位置エンコーディング 3手法の比較")
print("=" * 80)
print()
print(f"{'特徴':<24} {'Sinusoidal':<20} {'Learned':<20} {'RoPE'}")
print("-" * 80)
print(f"{'学習の必要性':<22} {'不要（固定）':<20} {'必要':<20} {'不要（固定）'}")
print(f"{'位置情報の種類':<22} {'絶対位置':<20} {'絶対位置':<20} {'相対位置'}")
print(f"{'長さの外挿':<22} {'理論上可能':<20} {'困難':<20} {'可能'}")
print(f"{'パラメータ数':<22} {'0':<20} {'max_len x d':<20} {'0'}")
print(f"{'適用方法':<22} {'加算':<20} {'加算':<20} {'回転（乗算）'}")
print(f"{'代表モデル':<22} {'Transformer原論文':<20} {'BERT, GPT-2':<20} {'LLaMA, GPT-NeoX'}")
print(f"{'提案年':<22} {'2017':<20} {'2018-2019':<20} {'2021'}")
print()
print("【選び方のガイドライン】")
print("  初学者/シンプルな実装 → Sinusoidal（実装が簡単、チューニング不要）")
print("  十分な訓練データあり  → Learned（データに適応できる）")
print("  長文処理/最新LLM     → RoPE（相対位置、長さ外挿に強い）")

---

## 6. 3つのアーキテクチャ分類

Transformerベースのモデルは、大きく3つのアーキテクチャに分類されます。

| 分類 | 代表モデル | Attention | 主なタスク |
|------|-----------|-----------|------------|
| **Encoder-only** | BERT | 双方向 | 分類、固有表現認識 |
| **Decoder-only** | GPT | 単方向（Causal） | テキスト生成 |
| **Encoder-Decoder** | T5, BART | Cross-Attention | 翻訳、要約 |

In [ ]:
# ============================================================
# Section 6: 3つのアーキテクチャの構造比較（図解）
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(20, 8))

def draw_architecture(ax, title, blocks, color):
    """アーキテクチャの構造を描画するヘルパー関数"""
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 12)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(title, fontsize=14, fontweight='bold', pad=20)
    
    y_pos = 1
    for block_name, block_color in blocks:
        rect = plt.Rectangle((1, y_pos), 8, 1.2, 
                             facecolor=block_color, edgecolor='black',
                             linewidth=1.5, alpha=0.8)
        ax.add_patch(rect)
        ax.text(5, y_pos + 0.6, block_name, ha='center', va='center',
                fontsize=10, fontweight='bold')
        y_pos += 1.5

# --- Encoder-only (BERT) ---
ax = axes[0]
blocks_enc = [
    ('Input Embedding + PE', '#a8d8ea'),
    ('Multi-Head Self-Attention\n(双方向)', '#fcbad3'),
    ('Add & LayerNorm', '#d4e157'),
    ('Feed-Forward Network', '#ffcc80'),
    ('Add & LayerNorm', '#d4e157'),
    ('[CLS] → 分類ヘッド', '#ce93d8'),
]
draw_architecture(ax, 'Encoder-only (BERT型)\n双方向Attention', blocks_enc, '#e3f2fd')
ax.text(5, 11.5, 'タスク: 分類、固有表現認識、穴埋め',
        ha='center', fontsize=10, style='italic', color='#1565c0')

# --- Decoder-only (GPT) ---
ax = axes[1]
blocks_dec = [
    ('Input Embedding + PE', '#a8d8ea'),
    ('Masked Multi-Head Attention\n(Causal / 単方向)', '#ef9a9a'),
    ('Add & LayerNorm', '#d4e157'),
    ('Feed-Forward Network', '#ffcc80'),
    ('Add & LayerNorm', '#d4e157'),
    ('Linear → Softmax\n(次トークン予測)', '#ce93d8'),
]
draw_architecture(ax, 'Decoder-only (GPT型)\nCausal Attention', blocks_dec, '#fce4ec')
ax.text(5, 11.5, 'タスク: テキスト生成、言語モデル',
        ha='center', fontsize=10, style='italic', color='#c62828')

# --- Encoder-Decoder (T5) ---
ax = axes[2]
blocks_encdec = [
    ('Encoder Self-Attention\n(双方向)', '#fcbad3'),
    ('Encoder FFN', '#ffcc80'),
    ('--- Cross-Attention ---\n(Decoder→Encoder)', '#b2dfdb'),
    ('Decoder Masked Attention\n(Causal)', '#ef9a9a'),
    ('Decoder FFN', '#ffcc80'),
    ('Linear → Softmax', '#ce93d8'),
]
draw_architecture(ax, 'Encoder-Decoder (T5型)\nCross-Attention', blocks_encdec, '#e8f5e9')
ax.text(5, 11.5, 'タスク: 翻訳、要約、QA',
        ha='center', fontsize=10, style='italic', color='#2e7d32')

plt.tight_layout()
plt.show()

print("【3つのアーキテクチャの使い分け】")
print("  Encoder-only:      入力を『理解』するタスク（分類、情報抽出）")
print("  Decoder-only:      テキストを『生成』するタスク（文章生成、対話）")
print("  Encoder-Decoder:   入力を理解して別の形で『出力』するタスク（翻訳、要約）")

In [ ]:
# ============================================================
# Causal Mask の実装と可視化
# ============================================================

def create_causal_mask(seq_len):
    """
    Causal Mask（下三角マスク）を生成する。
    
    未来のトークンを参照できないようにするマスク。
    Decoder-only モデル (GPT) で使用される。
    
    Parameters:
        seq_len: 系列長
    Returns:
        mask: [seq_len, seq_len] のマスク（1=マスク, 0=可視）
    """
    # 上三角部分を 1（マスク）にする
    mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1)
    return mask


# マスクの可視化
seq_len_mask = 8
tokens_mask = [f't{i}' for i in range(seq_len_mask)]

# 3種類のAttentionマスク
no_mask = torch.zeros(seq_len_mask, seq_len_mask)  # Encoder: 双方向
causal_mask = create_causal_mask(seq_len_mask)        # Decoder: Causal

# Encoder-Decoderの Cross-Attention マスクは通常マスクなし（Encoder全体を参照）

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Encoder: 双方向 ---
ax = axes[0]
# Attentionが可能な位置を表示
visible = 1 - no_mask.numpy()
sns.heatmap(visible, cmap='Greens', vmin=0, vmax=1,
            xticklabels=tokens_mask, yticklabels=tokens_mask,
            ax=ax, square=True, linewidths=0.5, linecolor='white',
            cbar=False, annot=True, fmt='.0f')
ax.set_title('Encoder (BERT型)\n双方向: 全トークンを参照可能', fontsize=13)
ax.set_xlabel('Key', fontsize=11)
ax.set_ylabel('Query', fontsize=11)

# --- Decoder: Causal ---
ax = axes[1]
visible_causal = 1 - causal_mask.numpy()
sns.heatmap(visible_causal, cmap='Oranges', vmin=0, vmax=1,
            xticklabels=tokens_mask, yticklabels=tokens_mask,
            ax=ax, square=True, linewidths=0.5, linecolor='white',
            cbar=False, annot=True, fmt='.0f')
ax.set_title('Decoder (GPT型)\nCausal: 過去のトークンのみ参照', fontsize=13)
ax.set_xlabel('Key', fontsize=11)
ax.set_ylabel('Query', fontsize=11)

# --- Cross-Attention ---
ax = axes[2]
enc_len = 5
dec_len = 8
cross_visible = np.ones((dec_len, enc_len))
sns.heatmap(cross_visible, cmap='Blues', vmin=0, vmax=1,
            xticklabels=[f'e{i}' for i in range(enc_len)],
            yticklabels=[f'd{i}' for i in range(dec_len)],
            ax=ax, square=True, linewidths=0.5, linecolor='white',
            cbar=False, annot=True, fmt='.0f')
ax.set_title('Cross-Attention (T5型)\nDecoder→Encoder: 全Encoder出力を参照', fontsize=13)
ax.set_xlabel('Encoder Key', fontsize=11)
ax.set_ylabel('Decoder Query', fontsize=11)

plt.tight_layout()
plt.show()

print("【マスクの意味】")
print("  1 = 参照可能（Attentionが計算される）")
print("  0 = マスク（-inf で softmax 後に 0 になる）")
print("\n  Causal Mask: t3 は t0, t1, t2, t3 のみ参照可能（t4以降は見えない）")
print("  → 自己回帰生成時に、未来の情報の漏洩を防ぐ")

---

## 7. Layer Normalization

### Pre-LN vs Post-LN の違い

TransformerにおけるLayer Normalizationの配置には2つの流派があります。

**Post-LN（原論文）:**
```
output = LayerNorm(x + Sublayer(x))
```

**Pre-LN（GPT-2以降で主流）:**
```
output = x + Sublayer(LayerNorm(x))
```

In [ ]:
# ============================================================
# Section 7: Pre-LN vs Post-LN の実装と比較
# ============================================================

class PostLNBlock(nn.Module):
    """
    Post-LN Transformer ブロック。
    output = LayerNorm(x + Sublayer(x))
    
    原論文の配置。深いネットワークで勾配が不安定になりやすい。
    """
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.ReLU(),
            nn.Linear(d_model * 4, d_model)
        )
        self.norm2 = nn.LayerNorm(d_model)
    
    def forward(self, x):
        # Post-LN: Norm は残差接続の後
        x = self.norm1(x + self.attn(x, x, x))
        x = self.norm2(x + self.ffn(x))
        return x


class PreLNBlock(nn.Module):
    """
    Pre-LN Transformer ブロック。
    output = x + Sublayer(LayerNorm(x))
    
    GPT-2以降で主流。学習が安定しやすい。
    """
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.ReLU(),
            nn.Linear(d_model * 4, d_model)
        )
        self.norm2 = nn.LayerNorm(d_model)
    
    def forward(self, x):
        # Pre-LN: Norm は残差接続の前
        x = x + self.attn(self.norm1(x), self.norm1(x), self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x


# --- 勾配流の比較実験 ---
print("=" * 60)
print("Pre-LN vs Post-LN: 勾配流の比較")
print("=" * 60)

n_layers_test = 12
d_model_test = 64
n_heads_test = 4

def measure_gradient_flow(block_class, n_layers, d_model, n_heads):
    """各層の勾配ノルムを測定する"""
    torch.manual_seed(42)
    
    # n_layers 層のモデルを構築
    layers = nn.ModuleList([block_class(d_model, n_heads) for _ in range(n_layers)])
    
    # 入力
    x = torch.randn(1, 10, d_model, requires_grad=True)
    
    # 順伝播
    hidden = x
    for layer in layers:
        hidden = layer(hidden)
    
    # ダミーの損失
    loss = hidden.sum()
    loss.backward()
    
    # 各層の重みの勾配ノルムを収集
    grad_norms = []
    for layer in layers:
        total_norm = 0
        for p in layer.parameters():
            if p.grad is not None:
                total_norm += p.grad.norm().item() ** 2
        grad_norms.append(math.sqrt(total_norm))
    
    return grad_norms

post_ln_grads = measure_gradient_flow(PostLNBlock, n_layers_test, d_model_test, n_heads_test)
pre_ln_grads = measure_gradient_flow(PreLNBlock, n_layers_test, d_model_test, n_heads_test)

# 可視化
fig, ax = plt.subplots(figsize=(10, 6))

layer_indices = list(range(1, n_layers_test + 1))
ax.plot(layer_indices, post_ln_grads, 'o-', label='Post-LN（原論文）',
        color='#e74c3c', linewidth=2, markersize=8)
ax.plot(layer_indices, pre_ln_grads, 's-', label='Pre-LN（GPT-2以降）',
        color='#3498db', linewidth=2, markersize=8)

ax.set_xlabel('層番号', fontsize=12)
ax.set_ylabel('勾配ノルム', fontsize=12)
ax.set_title(f'Pre-LN vs Post-LN: 各層の勾配ノルム ({n_layers_test}層)', fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
ax.set_yscale('log')

plt.tight_layout()
plt.show()

print("\n【Pre-LN vs Post-LN】")
print("  Post-LN: 深い層ほど勾配が大きくなりやすい（勾配爆発のリスク）")
print("  Pre-LN:  層間の勾配ノルムがより均一（学習が安定）")
print("\n  現代のTransformer (GPT-2, LLaMA等) はほぼ Pre-LN を採用")

---

## 8. Feed-Forward Network

### 8.1 FFN の役割

Transformerの各層は、Self-Attention + FFN で構成されます。
FFN は各トークンの表現を独立に非線形変換し、モデルの表現力を高めます。

**標準FFN:**
$$\text{FFN}(x) = \text{ReLU}(xW_1 + b_1)W_2 + b_2$$

**SwiGLU（現代的なFFN）:**
$$\text{SwiGLU}(x) = (\text{Swish}(xW_1) \odot xW_3) W_2$$

SwiGLU は LLaMA, PaLM 等の最新モデルで採用されています。

In [ ]:
# ============================================================
# Section 8: Feed-Forward Network の実装
# ============================================================

class StandardFFN(nn.Module):
    """標準的な2層FFN (ReLU活性化)"""
    def __init__(self, d_model, d_ff=None):
        super().__init__()
        if d_ff is None:
            d_ff = d_model * 4  # 慣例: 4倍に拡張
        self.w1 = nn.Linear(d_model, d_ff)
        self.w2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        # x: [batch, seq_len, d_model]
        return self.w2(self.relu(self.w1(x)))


class SwiGLUFFN(nn.Module):
    """
    SwiGLU Feed-Forward Network。
    
    SwiGLU(x) = (Swish(xW1) * xW3) W2
    
    Swish(x) = x * sigmoid(x) = SiLU(x)
    
    Gated Linear Unit の一種で、ゲーティング機構により
    より表現力の高い変換が可能。
    LLaMA, PaLM 等で採用。
    """
    def __init__(self, d_model, d_ff=None):
        super().__init__()
        if d_ff is None:
            # SwiGLU では 2/3 * 4d が一般的（パラメータ数を揃えるため）
            d_ff = int(d_model * 4 * 2 / 3)
        self.w1 = nn.Linear(d_model, d_ff)   # ゲート用
        self.w2 = nn.Linear(d_ff, d_model)   # 出力射影
        self.w3 = nn.Linear(d_model, d_ff)   # 値用
        self.silu = nn.SiLU()  # Swish = SiLU
    
    def forward(self, x):
        # ゲーティング: Swish(xW1) * xW3
        gate = self.silu(self.w1(x))
        value = self.w3(x)
        return self.w2(gate * value)


# --- 比較テスト ---
print("=" * 60)
print("Feed-Forward Network の比較")
print("=" * 60)

d_model_ffn = 64

standard = StandardFFN(d_model_ffn)
swiglu = SwiGLUFFN(d_model_ffn)

standard_params = sum(p.numel() for p in standard.parameters())
swiglu_params = sum(p.numel() for p in swiglu.parameters())

print(f"\nStandard FFN パラメータ数: {standard_params:,}")
print(f"SwiGLU FFN パラメータ数:  {swiglu_params:,}")

# テスト入力
torch.manual_seed(42)
test_input = torch.randn(1, 10, d_model_ffn)

with torch.no_grad():
    out_std = standard(test_input)
    out_swi = swiglu(test_input)

print(f"\n入力 shape: {test_input.shape}")
print(f"Standard FFN 出力 shape: {out_std.shape}")
print(f"SwiGLU FFN 出力 shape: {out_swi.shape}")

print("\n【FFN の役割】")
print("  Self-Attention: トークン間の情報交換（文脈の獲得）")
print("  FFN:            各トークンの表現を独立に変換（特徴の精緻化）")
print("\n  Transformerの容量の約2/3はFFNが占めている")

---

## 9. 小規模Transformerの実装

ここまでの部品を組み合わせて、**Encoder-only Transformer** を完全に実装し、
簡単なテキスト分類タスク（感情分析）で訓練します。

**モデル仕様:**
- 4層、4ヘッド、d_model=64
- Pre-LN 配置
- Sinusoidal 位置エンコーディング

In [ ]:
# ============================================================
# Section 9: Encoder-only Transformer の完全実装
# ============================================================

class TransformerEncoderBlock(nn.Module):
    """
    Transformer Encoder ブロック（Pre-LN）。
    
    構成:
    1. LayerNorm → Multi-Head Self-Attention → 残差接続
    2. LayerNorm → FFN → 残差接続
    """
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        # Pre-LN: Norm → Attention → Add
        normed = self.norm1(x)
        x = x + self.dropout(self.attn(normed, normed, normed, mask))
        # Pre-LN: Norm → FFN → Add
        x = x + self.ffn(self.norm2(x))
        return x


class TransformerClassifier(nn.Module):
    """
    Encoder-only Transformer によるテキスト分類モデル。
    
    構成:
    1. Token Embedding + Positional Encoding
    2. N層の Transformer Encoder ブロック
    3. [CLS] トークン（先頭トークン）の表現を使って分類
    
    Parameters:
        vocab_size: 語彙サイズ
        d_model: モデル次元数
        n_heads: Attentionヘッド数
        n_layers: Transformer層数
        d_ff: FFN中間次元数
        n_classes: 分類クラス数
        max_len: 最大系列長
        dropout: Dropout率
    """
    def __init__(self, vocab_size, d_model=64, n_heads=4, n_layers=4,
                 d_ff=256, n_classes=2, max_len=128, dropout=0.1):
        super().__init__()
        
        # Token Embedding
        self.token_emb = nn.Embedding(vocab_size, d_model)
        
        # Positional Encoding (Sinusoidal)
        self.pos_enc = SinusoidalPositionalEncoding(d_model, max_len)
        
        # Transformer Encoder 層
        self.layers = nn.ModuleList([
            TransformerEncoderBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        
        # 最終Layer Norm
        self.final_norm = nn.LayerNorm(d_model)
        
        # 分類ヘッド
        self.classifier = nn.Linear(d_model, n_classes)
        
        # Dropout
        self.dropout = nn.Dropout(dropout)
        
        # パラメータ初期化
        self._init_weights()
    
    def _init_weights(self):
        """重みの初期化"""
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
    
    def forward(self, x):
        """
        Parameters:
            x: トークンID列 [batch, seq_len]
        Returns:
            logits: 分類ロジット [batch, n_classes]
        """
        # Token Embedding + Positional Encoding
        x = self.token_emb(x)  # [batch, seq_len, d_model]
        x = self.pos_enc(x)
        x = self.dropout(x)
        
        # Transformer Encoder 層を通す
        for layer in self.layers:
            x = layer(x)
        
        # 最終Norm
        x = self.final_norm(x)
        
        # 先頭トークン（[CLS]相当）の表現を使って分類
        cls_output = x[:, 0, :]  # [batch, d_model]
        logits = self.classifier(cls_output)  # [batch, n_classes]
        
        return logits


# --- モデル構築 ---
print("=" * 60)
print("Encoder-only Transformer 分類モデル")
print("=" * 60)

# ハイパーパラメータ
VOCAB_SIZE = 200
D_MODEL = 64
N_HEADS = 4
N_LAYERS = 4
D_FF = 256
N_CLASSES = 2
MAX_LEN = 32

model = TransformerClassifier(
    vocab_size=VOCAB_SIZE, d_model=D_MODEL, n_heads=N_HEADS,
    n_layers=N_LAYERS, d_ff=D_FF, n_classes=N_CLASSES, max_len=MAX_LEN
).to(device)

# パラメータ数の表示
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nモデル構成:")
print(f"  語彙サイズ: {VOCAB_SIZE}")
print(f"  d_model: {D_MODEL}")
print(f"  ヘッド数: {N_HEADS} (d_k = {D_MODEL // N_HEADS})")
print(f"  層数: {N_LAYERS}")
print(f"  FFN中間次元: {D_FF}")
print(f"  分類クラス数: {N_CLASSES}")
print(f"\n総パラメータ数: {total_params:,}")
print(f"学習可能パラメータ数: {trainable_params:,}")

In [ ]:
# ============================================================
# 人工データセットの生成（感情分析タスク）
# ============================================================

def generate_sentiment_data(n_samples, vocab_size, seq_len, seed=42):
    """
    簡単な人工感情分析データセットを生成する。
    
    ルール:
    - ポジティブ (label=1): 特定のトークン (vocab の前半) が多い
    - ネガティブ (label=0): 特定のトークン (vocab の後半) が多い
    - ノイズとしてランダムなトークンも混ぜる
    
    これにより、Transformerがトークンの分布パターンを学習できる。
    """
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    X = []
    y = []
    
    # 特殊トークン: 0=[CLS], 1=[PAD]
    pos_tokens = list(range(2, vocab_size // 2))    # ポジティブトークン群
    neg_tokens = list(range(vocab_size // 2, vocab_size))  # ネガティブトークン群
    
    for i in range(n_samples):
        label = i % 2  # 交互に生成
        
        # [CLS] トークンで開始
        tokens = [0]  # [CLS]
        
        actual_len = np.random.randint(seq_len // 2, seq_len - 1)
        
        for _ in range(actual_len):
            if np.random.random() < 0.7:  # 70% はラベルに対応するトークン
                if label == 1:
                    tokens.append(np.random.choice(pos_tokens))
                else:
                    tokens.append(np.random.choice(neg_tokens))
            else:  # 30% はランダムノイズ
                tokens.append(np.random.randint(2, vocab_size))
        
        # パディング
        while len(tokens) < seq_len:
            tokens.append(1)  # [PAD]
        
        X.append(tokens[:seq_len])
        y.append(label)
    
    return torch.tensor(X), torch.tensor(y)


# データ生成
N_TRAIN = 1000
N_VAL = 200
SEQ_LEN = 32

X_train, y_train = generate_sentiment_data(N_TRAIN, VOCAB_SIZE, SEQ_LEN, seed=42)
X_val, y_val = generate_sentiment_data(N_VAL, VOCAB_SIZE, SEQ_LEN, seed=123)

print("=" * 60)
print("人工感情分析データセット")
print("=" * 60)
print(f"\n訓練データ: {X_train.shape[0]} サンプル")
print(f"検証データ: {X_val.shape[0]} サンプル")
print(f"系列長: {SEQ_LEN}")
print(f"ラベル分布 (訓練): {dict(zip(*np.unique(y_train.numpy(), return_counts=True)))}")
print(f"\nサンプル入力 (先頭10トークン): {X_train[0][:10].tolist()}")
print(f"サンプルラベル: {y_train[0].item()} ({'ポジティブ' if y_train[0].item() == 1 else 'ネガティブ'})")

In [ ]:
# ============================================================
# 訓練ループ
# ============================================================

def train_model(model, X_train, y_train, X_val, y_val, 
                n_epochs=30, batch_size=32, lr=1e-3):
    """
    Transformer分類モデルの訓練ループ。
    
    Returns:
        history: 訓練・検証の損失と精度の履歴
    """
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc': [], 'val_acc': []
    }
    
    n_train = len(X_train)
    n_val = len(X_val)
    
    for epoch in range(n_epochs):
        # --- 訓練 ---
        model.train()
        epoch_loss = 0
        correct = 0
        
        # ミニバッチ
        indices = torch.randperm(n_train)
        for i in range(0, n_train, batch_size):
            batch_idx = indices[i:i+batch_size]
            batch_x = X_train[batch_idx].to(device)
            batch_y = y_train[batch_idx].to(device)
            
            optimizer.zero_grad()
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            loss.backward()
            
            # 勾配クリッピング
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            epoch_loss += loss.item() * len(batch_idx)
            correct += (logits.argmax(dim=-1) == batch_y).sum().item()
        
        train_loss = epoch_loss / n_train
        train_acc = correct / n_train
        
        # --- 検証 ---
        model.eval()
        with torch.no_grad():
            val_logits = model(X_val.to(device))
            val_loss = criterion(val_logits, y_val.to(device)).item()
            val_acc = (val_logits.argmax(dim=-1) == y_val.to(device)).float().mean().item()
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:3d}/{n_epochs} | "
                  f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
                  f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")
    
    return history


print("=" * 60)
print("Transformer 分類モデルの訓練")
print("=" * 60)
print()

# モデルを再初期化
torch.manual_seed(42)
model = TransformerClassifier(
    vocab_size=VOCAB_SIZE, d_model=D_MODEL, n_heads=N_HEADS,
    n_layers=N_LAYERS, d_ff=D_FF, n_classes=N_CLASSES, max_len=MAXX_LEN
).to(device)

# 訓練実行
history = train_model(model, X_train, y_train, X_val, y_val,
                      n_epochs=30, batch_size=32, lr=1e-3)

print(f"\n最終検証精度: {history['val_acc'][-1]:.4f}")

In [ ]:
# ============================================================
# 学習曲線の可視化
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- 左: 損失 ---
ax = axes[0]
ax.plot(history['train_loss'], label='訓練損失', color='#e74c3c', linewidth=2)
ax.plot(history['val_loss'], label='検証損失', color='#3498db', linewidth=2)
ax.set_xlabel('エポック', fontsize=12)
ax.set_ylabel('損失 (Cross-Entropy)', fontsize=12)
ax.set_title('学習曲線: 損失', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# --- 右: 精度 ---
ax = axes[1]
ax.plot(history['train_acc'], label='訓練精度', color='#e74c3c', linewidth=2)
ax.plot(history['val_acc'], label='検証精度', color='#3498db', linewidth=2)
ax.set_xlabel('エポック', fontsize=12)
ax.set_ylabel('精度', fontsize=12)
ax.set_title('学習曲線: 精度', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(0.4, 1.05)

plt.suptitle('小規模Transformer (4層, 4ヘッド, d=64) による分類', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("【学習曲線の読み方】")
print("  訓練損失が下がり、検証損失も追従 → 良好な学習")
print("  訓練損失 << 検証損失 → 過学習の兆候")
print("  検証精度が頭打ち → モデル容量やデータ量の限界")

---

## 10. まとめ・チートシート・よくある間違い

### 10.1 まとめ

このノートブックでは、NLPのためのTransformerアーキテクチャを体系的に学びました。

1. **Self-Attention**: Query/Key/Valueの仕組みとスケーリングの重要性
2. **Multi-Head Attention**: 複数ヘッドで異なる言語的関係を並列に捉える
3. **位置エンコーディング**: Sinusoidal / Learned / RoPE の3手法の比較
4. **3つのアーキテクチャ**: Encoder-only, Decoder-only, Encoder-Decoder
5. **Layer Normalization**: Pre-LN vs Post-LN の安定性の違い
6. **FFN**: 標準FFNとSwiGLUの比較
7. **完全実装**: Encoder-only Transformer でテキスト分類

### 10.2 チートシート

| コンポーネント | 数式 | 役割 |
|--------------|------|------|
| Scaled Dot-Product Attention | $\text{softmax}(QK^T/\sqrt{d_k})V$ | トークン間の関係を計算 |
| Multi-Head Attention | $\text{Concat}(\text{head}_1,...,\text{head}_h)W^O$ | 複数の観点で注目 |
| Sinusoidal PE | $\sin(pos/10000^{2i/d})$ | 固定的な位置情報 |
| RoPE | 回転行列で相対位置 | 長さ外挿に強い |
| Pre-LN | $x + F(\text{LN}(x))$ | 安定した学習 |
| SwiGLU | $\text{Swish}(xW_1) \odot xW_3$ | 表現力の高いFFN |

### 10.3 よくある間違い

#### 間違い 1: スケーリングを忘れる

$QK^T$ を $\sqrt{d_k}$ で割らないと、次元数が大きい場合にsoftmaxの出力がほぼone-hotになり、勾配が消失します。必ずスケーリングを行いましょう。

#### 間違い 2: Causal Mask の方向を間違える

Causal Maskは**上三角**をマスク（-inf）にします。つまり、各トークンは自分自身と過去のトークンのみ参照可能です。方向を逆にすると、未来の情報が漏洩します。

#### 間違い 3: Pre-LN と Post-LN を混同する

- Post-LN: `LN(x + F(x))` — 原論文の配置
- Pre-LN: `x + F(LN(x))` — 現代の主流

Pre-LN の方が学習が安定するため、特に深いモデルではPre-LNを使いましょう。

#### 間違い 4: d_model を n_heads で割れないサイズにする

Multi-Head Attentionでは `d_k = d_model / n_heads` です。d_modelがn_headsで割り切れないとエラーになります。一般的な組み合わせ: d=512/h=8, d=768/h=12, d=1024/h=16。

#### 間違い 5: 位置エンコーディングの加算と連結を混同する

Sinusoidal PE と Learned PE は埋め込みに**加算**します。連結（concatenation）ではありません。次元数が変わってしまうので注意。RoPEは乗算的に適用されます。

---

## 11. 自己評価クイズ

---

**Q1.** Scaled Dot-Product Attention で $\sqrt{d_k}$ で割る理由を説明してください。

<details>
<summary>回答を表示</summary>

Query と Key のドット積 $QK^T$ の値は、次元数 $d_k$ が大きくなると分散が大きくなります（各要素が独立なら分散が $d_k$ に比例）。これにより softmax の入力が極端に大きくなり、出力がほぼ one-hot（1つの要素だけが1に近い）になってしまいます。$\sqrt{d_k}$ で割ることで分散を1に正規化し、softmax が適度に滑らかな分布を出力するようにします。これにより勾配が消失しにくくなり、学習が安定します。

</details>

---

**Q2.** RoPE が Sinusoidal PE より長文処理に強い理由を説明してください。

<details>
<summary>回答を表示</summary>

Sinusoidal PE は**絶対位置**をエンコードするため、訓練時に見た最大長を超えると未知の位置表現となります。一方、RoPE は**相対位置**をドット積に埋め込むため、2つのトークン間の距離が重要であり、絶対位置には依存しません。そのため、訓練時より長い系列でも相対的な位置関係は保たれ、外挿（length extrapolation）が可能です。

</details>

---

**Q3.** Encoder-only（BERT型）とDecoder-only（GPT型）の最も本質的な違いは何ですか？

<details>
<summary>回答を表示</summary>

最も本質的な違いは **Attention の方向** です。Encoder-only は **双方向（Bidirectional）** Attention で、各トークンが系列の全トークンを参照できます。Decoder-only は **単方向（Causal/Unidirectional）** Attention で、各トークンは自分自身と過去のトークンのみ参照可能です。この違いにより、Encoder-only は入力全体の理解（分類等）に適し、Decoder-only は左から右への逐次生成に適しています。

</details>

---

**Q4.** Multi-Head Attention で複数のヘッドを使う利点を2つ挙げてください。

<details>
<summary>回答を表示</summary>

1. **異なる言語的関係の並列学習**: 各ヘッドが異なる射影空間でAttentionを計算するため、構文的関係（主語-動詞）、意味的関係（共参照）、位置的近接など、複数の種類の関係を同時に捉えられます。

2. **表現の多様性**: 単一のヘッドでは1つの「観点」からしかAttentionを計算できませんが、複数ヘッドにより多様な注目パターンを学習でき、モデルの表現力が大幅に向上します。また、各ヘッドの次元が小さくなるため、計算コストは単一ヘッドと変わりません。

</details>

---

**Q5.** SwiGLU が標準的な ReLU FFN より優れている理由を説明してください。

<details>
<summary>回答を表示</summary>

SwiGLU はゲーティング機構（Gated Linear Unit）を導入しており、$\text{Swish}(xW_1) \odot xW_3$ の形で情報の流れを制御します。標準的な ReLU FFN（$\text{ReLU}(xW_1)W_2$）と比較して:

1. **Swish活性化**: ReLU と異なり、負の領域でも微小な勾配を持つため、情報の損失が少ない
2. **ゲーティング**: $xW_3$ によるゲートが、どの特徴を通すかを学習的に制御できる
3. **実験的に優れた性能**: LLaMA, PaLM 等の大規模言語モデルで、同じパラメータ数で標準FFNより一貫して高い性能が報告されている

</details>

---

## 参考文献

1. Vaswani, A., et al. (2017). *Attention is All You Need*. NeurIPS.
2. Devlin, J., et al. (2019). *BERT: Pre-training of Deep Bidirectional Transformers*. NAACL.
3. Radford, A., et al. (2019). *Language Models are Unsupervised Multitask Learners*. OpenAI.
4. Su, J., et al. (2021). *RoFormer: Enhanced Transformer with Rotary Position Embedding*. arXiv.
5. Shazeer, N. (2020). *GLU Variants Improve Transformer*. arXiv.
6. Xiong, R., et al. (2020). *On Layer Normalization in the Transformer Architecture*. ICML.